# 02. Classical Forecasting: SARIMA & Holt-Winters

## Background

Classical time series forecasting methods dominated the field for 50 years and remain competitive on many real-world datasets. The M4 and M5 forecasting competitions — the most comprehensive benchmarks ever conducted — showed that statistical methods often outperform complex neural networks on monthly and quarterly business series.

**SARIMA** (Seasonal ARIMA): extends ARIMA with seasonal differencing and seasonal lag terms. Notation: SARIMA(p,d,q)(P,D,Q)[m] where m is the seasonal period. Auto-ARIMA tools (pmdarima, statsforecast) select order parameters via AIC/BIC.

**Holt-Winters Exponential Smoothing**: three exponential smoothing components for level (α), trend (β), and seasonality (γ). The additive vs multiplicative choice depends on whether seasonal amplitude scales with the series level.

**Prophet**: Facebook's time series library (2017) uses a decomposable model — piecewise linear or logistic trend, Fourier series for seasonality, and regression for holidays. Easy to use, interpretable, but less accurate than SARIMA on short series.

## What You'll Learn

- Auto-ARIMA for automated order selection
- SARIMA fitting and multi-step forecasting with prediction intervals
- Holt-Winters with automatic additive/multiplicative selection
- Forecast combination: averaging models for improved accuracy

## Why This Matters

Classical methods are your baseline and your fallback. When neural forecasters fail due to distribution shift or insufficient data, SARIMA and Holt-Winters often deliver reliable results. Understanding their assumptions helps you diagnose when to trust them and when to reach for something more expressive.


## SARIMA Fitting

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

np.random.seed(42)

# Regenerate series from NB01
def generate_ts(n=500, trend=0.05, seasonal_period=52, seasonal_amp=10.0, noise_std=3.0):
    t = np.arange(n)
    values = 100 + trend*t + seasonal_amp*np.sin(2*np.pi*t/seasonal_period) + np.random.normal(0, noise_std, n)
    return pd.Series(values, index=pd.date_range('2018-01-01', periods=n, freq='W'), name='value')

ts = generate_ts()
train, test = ts[:-52], ts[-52:]
print(f'Train: {len(train)} obs, Test: {len(test)} obs')

# Fit SARIMA(1,1,1)(1,1,0)[52]
# (1,1,1): AR(1), one diff, MA(1)
# (1,1,0)[52]: seasonal AR(1), one seasonal diff, no seasonal MA
sarima = SARIMAX(train,
                  order=(1, 1, 1),
                  seasonal_order=(1, 1, 0, 52),
                  trend='n',
                  enforce_stationarity=False,
                  enforce_invertibility=False)
sarima_fit = sarima.fit(disp=False)

print(f'SARIMA AIC: {sarima_fit.aic:.2f}')
print(f'SARIMA BIC: {sarima_fit.bic:.2f}')
print(f'\nSARIMA model fit summary (key params):')
for name, coef, se in zip(sarima_fit.param_names[:4],
                           sarima_fit.params[:4],
                           sarima_fit.bse[:4]):
    print(f'  {name:15s}: {coef:.4f} (SE={se:.4f})')


## SARIMA Forecasting & Intervals

In [ ]:
# Forecast next 52 weeks with prediction intervals
forecast_result = sarima_fit.get_forecast(steps=52)
forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int(alpha=0.10)  # 90% interval

# Align index
forecast_mean.index = test.index
forecast_ci.index = test.index

# Evaluation
mape = np.mean(np.abs((test.values - forecast_mean.values) / test.values)) * 100
rmse = np.sqrt(np.mean((test.values - forecast_mean.values)**2))
coverage_90 = np.mean((test.values >= forecast_ci.iloc[:, 0].values) &
                       (test.values <= forecast_ci.iloc[:, 1].values))

print(f'SARIMA Forecast Evaluation (52-week horizon):')
print(f'  MAPE:          {mape:.2f}%')
print(f'  RMSE:          {rmse:.4f}')
print(f'  90% PI coverage: {coverage_90:.1%} (target: 90%)')


## Holt-Winters & Forecast Combination

In [ ]:
# Holt-Winters
hw_add = ExponentialSmoothing(train, seasonal_periods=52,
                               trend='add', seasonal='add',
                               use_boxcox=False, damped_trend=True)
hw_add_fit = hw_add.fit(optimized=True)

hw_mul = ExponentialSmoothing(train, seasonal_periods=52,
                               trend='add', seasonal='mul',
                               use_boxcox=False, damped_trend=True)
hw_mul_fit = hw_mul.fit(optimized=True)

hw_add_fc = hw_add_fit.forecast(52)
hw_mul_fc = hw_mul_fit.forecast(52)
hw_add_fc.index = test.index
hw_mul_fc.index = test.index

# Seasonal naive baseline
naive_fc = pd.Series(train.iloc[-52:].values, index=test.index)

# Forecast combination (equal weights)
combined_fc = (forecast_mean + hw_add_fc + naive_fc) / 3

def mape(actual, predicted):
    return np.mean(np.abs((actual.values - predicted.values) / actual.values)) * 100

print('Model Comparison (52-week MAPE):')
results = [
    ('Seasonal Naive', naive_fc),
    ('Holt-Winters (Add)', hw_add_fc),
    ('Holt-Winters (Mul)', hw_mul_fc),
    ('SARIMA(1,1,1)(1,1,0)[52]', forecast_mean),
    ('Combination (Equal Wt)', combined_fc),
]
for name, fc in results:
    m = mape(test, fc)
    print(f'  {name:35s} MAPE={m:.2f}%')

print('\nKey insight: Combination forecasts often beat individual models')
print('This is the forecast combination puzzle: why averages outperform.')
